In [1]:
# env 파일 읽어오기

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

MODEL = 'gpt-6-luna'

#### Decision API
- AI 에게 긴 답변을 생성하도록 하는 API가 아니라 주어진 데이터를 평가해서 어플리케이션이 바로 사용할 수 있게 판단 결과를 반환
    - model
    - input : 판단의 근거가 되는 데이터
    - question : 입력 데이터에 대해 수행할 판단 질문
        - predicate : 특정 조건이 참 or 거짓인지 판단
        - choice : 선택지 제공
        - score : 점수

In [3]:
from openai import OpenAI

client = OpenAI()

decision = client.decisions.create(
    model=MODEL,
    input="고객 문의 : 결제가 두 번 청구됐어요.",
    questions=[
        {
            "type":"choice",
            "name":"department",
            "instructions":"어느 부서에서 처리해야 하는지 문의인가요?",
            "choices":[
                {
                    "value":"billing",
                    "description":"결제, 청구, 환불 관련 문의"
                },
                {
                    "value":"technical",
                    "description":"서비스 사용 및 기술적 문제"
                },                
                {
                    "value":"shipping",
                    "description":"배송 및 배송 추적 관련 문의"
                },               
                {
                    "value":"other",
                    "description":"그 외 문의"
                },
            ]
        }
    ]
)


In [4]:
answer = decision.answers[0]
print(answer)
print(answer.choice)
print(answer.confidence)

AnswerAnswerResourceChoice(choice='billing', confidence=1.0, name='department', probabilities=[AnswerAnswerResourceChoiceProbability(probability=1.0, value='billing'), AnswerAnswerResourceChoiceProbability(probability=0.0, value='technical'), AnswerAnswerResourceChoiceProbability(probability=0.0, value='shipping'), AnswerAnswerResourceChoiceProbability(probability=0.0, value='other')], type='choice')
billing
1.0


In [17]:
# 이미지 판단

from openai import OpenAI
import base64

client = OpenAI()

def encode_image(image_path):
    with open(image_path, "rb")as f:
        return base64.b64encode(f.read()).decode("ascii")

image_path = "./data/dented-mug.png"

# 이미지 로드
base64_image = encode_image(image_path)

decision = client.decisions.create(
    model=MODEL,
    input=[
        {"role":"user", "content":[
            {"type":"input_text", "text":"Is there a problem with the item in the photo?"},
            {"type":"input_image", "image_url":f"data:image/png;base64,{base64_image}"},
        ]}
    ],
    questions=[
        {
            "type":"predicate",
            "name":"visible_damage",
            "instructions":[
                {"type":"input_text","text": "제품에 균열, 찢어짐, 찌그러짐 등등 눈에 띄는 손상이 있나요?"},
                {"type":"input_text","text": "그림자나 포장의 손상은 고려하지 마세요"},               
            ],
        }
    ]
)

answer = decision.answers[0]
print(answer)
print(answer.probability)
print(answer.type)


AnswerAnswerResourcePredicate(name='visible_damage', probability=1.0, type='predicate')
1.0
predicate


In [18]:
# 이미지 판단

from openai import OpenAI
import base64

client = OpenAI()

def encode_image(image_path):
    with open(image_path, "rb")as f:
        return base64.b64encode(f.read()).decode("ascii")

image_path = "./data/dimpled-tin.png"

# 이미지 로드
base64_image = encode_image(image_path)

decision = client.decisions.create(
    model=MODEL,
    input=[
        {"role":"user", "content":[
            {"type":"input_text", "text":"Is there a problem with the item in the photo?"},
            {"type":"input_image", "image_url":f"data:image/png;base64,{base64_image}"},
        ]}
    ],
    questions=[
        {
            "type":"predicate",
            "name":"visible_damage",
            "instructions":[
                {"type":"input_text","text": "제품에 균열, 찢어짐, 찌그러짐 등등 눈에 띄는 손상이 있나요?"},
                {"type":"input_text","text": "그림자나 포장의 손상은 고려하지 마세요"},               
            ],
        }
    ]
)

answer = decision.answers[0]
print(answer)
print(answer.probability)
print(answer.type)

AnswerAnswerResourcePredicate(name='visible_damage', probability=0.86, type='predicate')
0.86
predicate


In [20]:
decision = client.decisions.create(
    model=MODEL,
    input=[
        {"role":"user", "content":[
            {"type":"input_text", "text":"고객문의와 주문 정보를 비교하세요"},
            {"type":"input_json", "data":{
                "ticked":{"body":"같은 금액이 두 번 결제됐어요."},
                "other":{"charges":[49000,49000]}
            }},
        ]}
    ],
    questions=[
        {
            "type":"predicate",
            "name":"duplicate_charge",
            "instructions":"주문 정보상 중복 결제가 발생한 것이 맞나요?"
        }
    ]
)

answer = decision.answers[0]
print(answer)


AnswerAnswerResourcePredicate(name='duplicate_charge', probability=0.99, type='predicate')


In [21]:
decision = client.decisions.create(
    model=MODEL,
    input="Edge에서는 내보내기가 실패하지만 Chrome에서는 정상적으로 동작합니다.",
    questions=[
        {
            "type":"score",
            "name":"severity",
            "instructions":"해당 문제의 심각성은 얼마인지 점수로 알려주세요.",
            "levels":[
                {
                    "label":"Cosmetic",
                    "description":"외관상의 문제일 분, 기능상 손실은 없습니다."
                },
                {
                    "label":"Workaround available",
                    "description":"특정 작업이 실패하지만, 다른 우회 방법이 있습니다."
                },
                {
                    "label":"Fully blocked",
                    "description":"우회 방법도 없이 작업이 완전히 막힌 상태입니다."
                }
            ]
        }
    ]
)

answer = decision.answers[0]
print(answer.score)

0.99


In [23]:
input="""
결제가 계속 실패하고 있습니다.
오늘까지 결제하지 못하면 서비스를 이용할 수 없어요.
빠르게 해결해 주세요.
"""

decision = client.decisions.create(
    model=MODEL,
    input=input,
    questions=[
        {
            "type":"choice",
            "name":"department",
            "instructions":"어느 부서에서 처리해야 하는지 문의인가요?",
            "choices":[
                {
                    "value":"billing",
                    "description":"결제, 청구, 환불 관련 문의"
                },
                {
                    "value":"technical",
                    "description":"서비스 사용 및 기술적 문제"
                },                
                {
                    "value":"shipping",
                    "description":"배송 및 배송 추적 관련 문의"
                },               
                {
                    "value":"other",
                    "description":"그 외 문의"
                },
            ]
        },
        {        
            "type":"predicate",
            "name":"urgent",
            "instructions":"긴급하게 처리해야 하는 문의인가요?"
        },
        {
            "type":"choice",
            "name":"sentiment",
            "instructions":"고객의 감정 상태를 분류하세요",
            "choices":[
                {'value':"positive", "description":"긍정"},
                {'value':"negative", "description":"부정"},
                {'value':"neutral", "description":"중립"}
            ]
        }
    ]
)

answer = decision.answers[0]
print(answer)

AnswerAnswerResourceChoice(choice='billing', confidence=1.0, name='department', probabilities=[AnswerAnswerResourceChoiceProbability(probability=1.0, value='billing'), AnswerAnswerResourceChoiceProbability(probability=0.0, value='technical'), AnswerAnswerResourceChoiceProbability(probability=0.0, value='shipping'), AnswerAnswerResourceChoiceProbability(probability=0.0, value='other')], type='choice')


In [27]:
for answer in decision.answers:
    print("이름", answer.name)
    print("유형", answer.type)
    print("답변", answer)
    print()

이름 department
유형 choice
답변 AnswerAnswerResourceChoice(choice='billing', confidence=1.0, name='department', probabilities=[AnswerAnswerResourceChoiceProbability(probability=1.0, value='billing'), AnswerAnswerResourceChoiceProbability(probability=0.0, value='technical'), AnswerAnswerResourceChoiceProbability(probability=0.0, value='shipping'), AnswerAnswerResourceChoiceProbability(probability=0.0, value='other')], type='choice')

이름 urgent
유형 predicate
답변 AnswerAnswerResourcePredicate(name='urgent', probability=1.0, type='predicate')

이름 sentiment
유형 choice
답변 AnswerAnswerResourceChoice(choice='negative', confidence=1.0, name='sentiment', probabilities=[AnswerAnswerResourceChoiceProbability(probability=0.0, value='positive'), AnswerAnswerResourceChoiceProbability(probability=1.0, value='negative'), AnswerAnswerResourceChoiceProbability(probability=0.0, value='neutral')], type='choice')

